# Session 3 · Part 1 — Evaluate pointwise prediction accuracy

**Goal:** quantify how well inferred abundance tracks measured abundance for every protein. Pearson
correlation emphasizes linear agreement; Spearman correlation emphasizes rank agreement. Neither checks
whether the predicted tissue pattern is spatially coherent—that is Part 2.

This is an evaluation notebook: it requires the matching official Breast RNA/ADT assets and never pairs
the committed predictions with any other observations.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


## 1. Align observed and predicted proteins and read provenance


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dgat_tutorial.checkpoints import preferred_prediction_path
from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.dgat import load_prediction_metadata, load_prediction_table
from dgat_tutorial.evaluation import protein_correlations
from dgat_tutorial.plotting import plot_correlation_bar

dataset = load_tutorial_data(paths.raw_data)
prediction_path = preferred_prediction_path(paths)
predicted = load_prediction_table(str(prediction_path))
metadata = load_prediction_metadata(prediction_path)
if metadata:
    print(f"Prediction method: {metadata['method']}")
    print(f"Evaluation note: {metadata['evaluation_note']}")

common_spots = dataset.proteins.index.intersection(predicted.index)
common_proteins = dataset.proteins.columns.intersection(predicted.columns)
if common_spots.empty or common_proteins.empty:
    raise ValueError("Observed and predicted tables must share spot IDs and canonical protein names.")
observed = dataset.proteins.loc[common_spots, common_proteins]
predicted = predicted.loc[common_spots, common_proteins]
correlations = protein_correlations(observed, predicted)
correlations


### Figure 10 — Accuracy across the complete protein panel


In [ ]:
ax = plot_correlation_bar(correlations, metric="pearson")
ax.set_title("Per-protein pointwise accuracy")
correlation_bar_path = paths.figures / "session03_prediction_correlations.png"
plt.tight_layout(); plt.savefig(correlation_bar_path, dpi=160, bbox_inches="tight"); plt.show()


### Figure 11 — Observed versus predicted abundance for representative proteins


In [ ]:
ranked = correlations.sort_values("pearson")
representative = list(dict.fromkeys([
    ranked.iloc[-1]["protein"], ranked.iloc[len(ranked)//2]["protein"], ranked.iloc[0]["protein"]
]))
fig, axes = plt.subplots(1, len(representative), figsize=(4 * len(representative), 3.6), squeeze=False)
for ax, protein in zip(axes.ravel(), representative):
    ax.scatter(observed[protein], predicted[protein], s=10, alpha=0.45, color="#3b7a78")
    ax.set(xlabel="observed", ylabel="predicted", title=f"{protein}\nPearson={observed[protein].corr(predicted[protein]):.2f}")
scatter_path = paths.figures / "session03_observed_vs_predicted_scatter.png"
fig.tight_layout(); fig.savefig(scatter_path, dpi=160, bbox_inches="tight"); plt.show()


**How to read it:** a high correlation with a compressed prediction range can still underestimate
biological extremes. Outliers may be technical, but they can also identify rare regions worth inspecting
spatially. Avoid summarizing a 31-protein panel with only its mean correlation.


In [ ]:
table_path = paths.results / "session03_prediction_correlations.csv"
correlations.to_csv(table_path, index=False)
manifest = write_checkpoint(
    "3.1", [table_path, correlation_bar_path, scatter_path],
    summary={"spots": len(common_spots), "proteins_evaluated": len(correlations)}, start=paths.root,
)
print(f"Checkpoint written: {manifest}")


## Check

Name a high-, middle-, and low-performing protein and describe whether errors look like noise, range
compression, or systematic bias. Carry those examples into the spatial evaluation.
